In [9]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math


# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# returns 1, 2, or 3 with equal probability using quantum measurements
def random():
  random_transform_matrix = [[-math.sqrt(2)/math.sqrt(3), 1/math.sqrt(3)],[1/math.sqrt(3),math.sqrt(2)/math.sqrt(3)]]
  random_transform = Operator(random_transform_matrix)

  # create a qubit with state 1/root3|0> + 2/root3|1>

  ran = QuantumCircuit(1)
  ran.x(0)
  ran.unitary(random_transform, [0])

  # measure the qubit
  ran.measure_all()

  backend = BasicSimulator()
  compiled = transpile(ran, backend)
  job_sim = backend.run(compiled, shots=1)
  result_sim = job_sim.result()
  counts = result_sim.get_counts(compiled)

  # since it has a 1/3 chance of being 0
  if (counts.get("0",0)) == 1:
    return 1

  # since there was 2/3 chance of getting 1
  # the 2/3 probability is split with a 50/50 measurement,
  # giving outcomes 2 and 3 with probability 1/3 each
  ran2 = QuantumCircuit(1)
  ran2.h(0)
  ran2.measure_all()
  backend = BasicSimulator()
  compiled = transpile(ran2, backend)
  job_sim = backend.run(compiled, shots=1)
  result_sim = job_sim.result()
  counts = result_sim.get_counts(compiled)

  if (counts.get("0",0))== 1:
    return 2
  return 3


In [12]:
# Useful constants
root2 = math.sqrt(2)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2)


W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]

W_transform = Operator(W_transform_matrix)

V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]

V_transform = Operator(V_transform_matrix)

N = 100

repetitions = math.ceil((9 * N) /2)

shared_key = []
average = {'XV': 0 , 'XW': 0 , 'ZW': 0, 'ZV': 0 }
totals = {'XV': 0 , 'XW': 0 , 'ZW': 0, 'ZV': 0 }
counters = {'XV': 0 , 'XW': 0 , 'ZW': 0, 'ZV': 0 }

# creates an entangled pair of bits in right state
# these represent a pair of the bits alice and bob will get
def entangledPair():
    # Create 2 qubits |00>
    q = QuantumCircuit(2,2)

    # put qubit 0 into superposition
    # 1/root2(|00> + |01>)
    q.h(0)

    # create entanglement
    # 1/root2(|00> + |11>)
    q.cx(0,1)

    # 1/root2(|01> + |10>)
    q.x(0)

    # 1/root2(|01> - |10>)
    q.z(1)

    # used this to check it
    state = Statevector.from_int(0, 4)
    # Apply the circuit to the state
    state = state.evolve(q)
    state.draw("latex")

    return q

# For simulation purposes we perform the operations and measurements
# one pair at a time. In the real protocol Alice and Bob would receive
# many entangled qubits, measure them in their chosen bases, and then
# publicly share their basis choices afterwards
#
# Since we assume there is no attacker, Alice and Bob will obtain
# identical key bits (after Bob flips his bit where required), so we
# only store Alice's version of the shared key

for rep in range(repetitions):

  # Alice and Bob each receive one qubit of an entangled pair in state
  # they will receive their bits after attacker has interfered
  # q1 is Alice's bit q0 is Bob's bit
  sim = entangledPair()


  # Attacker method 1:
  # Measure Bob's qubit before he receives it
  # This collapses the entangled state, destroying the quantum correlations
  # Optionally, the attacker could measure Alice's qubit instead:
  # sim.measure(1, 1)  # Alice's qubit
  # Bob's qubit
  sim.measure(0,0)

  # Attacker method 2:
  # Apply a single-qubit operator (X or Z) or apply a Hadamard (H) to Bob's qubit
  # This does not completely break the entanglement, but the CHSH value |S| is significantly reduced
  # sim.x(0)

  # Attacker method 3:
  # Measure Bob's qubit in different basis (X basis here) before he receives it
  # By measuring it, it collapses the entangled state, destroying the quantum correlations
  # sim.h(0)
  # sim.measure(0, 0)

  # Alice chooses an operator Ai randomly from her set of three, with probability 1/3 each
  i = random()

  # Alice measures operator Ai on her qubit.
  if i == 1:
      # measure in X basis - Diagonal basis
      # basically apply h
      sim.h(1)
  elif i == 2:
      # measure in W basis
      # basically apply BO
      sim.unitary(W_transform,[1])
  else:
      # measure in Z basis - Standard Basis
      # do nothing
      pass

  # Bob chooses an operator Bj randomly from his set of three, with probability 1/3
  j = random()

  # Bob measures operator Bj on his qubit
  if j == 1:
      # measure in W basis
      # basically apply BO
      sim.unitary(W_transform, [0])
  elif j == 2:
      # measure in Z basis - Standard Basis
      # do nothing
      pass
  else:
      # measure in V basis
      # basically apply B1
      sim.unitary(V_transform, [0])

  # In the real protocol Alice / Bob measures all qubits after distribution.
  # Here we measure one pair at a time for simplicity in the simulation.
  sim.measure(0,0)
  sim.measure(1,1)

  backend = BasicSimulator()
  compiled = transpile(sim, backend)
  job_sim = backend.run(compiled, shots=1)
  result_sim = job_sim.result()
  counts = result_sim.get_counts(compiled)
  count00 = counts.get("00",0)
  count01 = counts.get("01",0)
  count10 = counts.get("10",0)
  count11 = counts.get("11",0)

  # Attacker method 4:
  # When Alice and Bob share their sequence of operator choices (basis),
  # the attacker intercepts and sends random operator choices for Alice instead
  # This ensures that Bob uses the wrong bases,
  # and this reduces the CHSH value |S|
  # i = quantum_random()


  # Here Alice and Bob share which basis they
  # measured in. Based on this information,
  # they decide whether to use the result for the shared key,
  # use it to calculate |S|, or discard it


  # If they measured in the same basis then the result will be used for the shared_key
  # Bob will invert his bit as agreed when creating the key so that they have the same key
  if (i == 2 and j == 1 ) or( i == 3 and j == 2 ):
    # if alices bit is 1 then 1 will be used in key else 0
    shared_key.append(count10)
    continue

  # For measurement combinations used to calculate |S|
  # add the result to the total for that pair of operations
  # and record how many times each combination occurs.
  if (i == 1 and j == 1):
    totals['XW'] += (count00 - count01 - count10 + count11)
    counters['XW'] += 1

  elif (i == 1 and j == 3):
    totals['XV'] += (count00 - count01 - count10 + count11)
    counters['XV'] += 1

  elif (i == 3 and j == 1):
    totals['ZW'] += (count00 - count01 - count10 + count11)
    counters['ZW'] += 1

  elif (i == 3 and j == 3):
    totals['ZV'] += (count00 - count01 - count10 + count11)
    counters['ZV'] += 1

# Use the totals and occurrences to calculate the average for each pair
for m in totals:
  if counters[m] > 0:
    average[m] = totals[m]/ counters[m]

# Find |S| using the averages and formula
# if |S| is close to 2root2 then can conclude that there was no attacker

# We should find that |S| will be <= 2 because the attacker has interfered with every entangled pair
result = abs(average['XW'] - average['XV'] + average['ZW'] + average['ZV'])
print(f"Final result : {result:.3f}")
print(f"Expected result (with attacker): <=2")

print(f"Expected result(2 Root 2) (without attacker): {2*root2:.3f} ")
print(f"key : {len(shared_key)}")



Final result : 1.406
Expected result (with attacker): <=2
Expected result(2 Root 2) (without attacker): 2.828 
key : 100
